In [2]:
%load_ext autoreload
%autoreload 2

In [65]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
print("PROJECT ROOT ==>", PROJECT_ROOT)

sys.path.append(str(PROJECT_ROOT))
from src.mesh_fun import (
    load_segmentation,load_pred_landmarks,visualize_pred_and_gt,visualize,load_mesh,
    color_mesh_by_groups,create_landmark_spheres,load_landmarks,
    load_gt_landmarks,save_screenshot
)
from src.utils_fun import build_perturbation_label
from src.landmark_eval import( 
    compute_all_statistics,
    evaluate_single_scan_from_landmarks,
    quality_profile_for_scan,
select_examples_by_quantile, 
compute_profiles_for_scans)
from src.var_constants import CATEGORIES,TOOTH_GROUP_PALETTE,LANDMARK_PALETTE,TOOTH_TO_GROUP


PROJECT ROOT ==> C:\Users\aless\Tesi_Msc\multi_agent_evaluation_on_intraoral_3D_scans


In [4]:
#PROJECT_ROOT = Path(__file__).resolve().parent
DATA_ROOT = PROJECT_ROOT / "dataset"
SCANS = DATA_ROOT / "toothinstancenet_input"
GT_ROOT = SCANS
PRED_CSV = DATA_ROOT / "model_predictions" / "ynlab" / "predictions.csv"
SCREENSHOT_DIR = DATA_ROOT / "screenshot_scans" / "raw"

In [83]:
scan_name='AKBDPB4C_upper'
scan_name='RZ69JRI9_lower'

In [84]:
screenshot_mesh_path =SCREENSHOT_DIR / scan_name
mesh_path = SCANS / f"{scan_name}.obj"
gt_json = GT_ROOT / f"{scan_name}__kpt.json"
seg_path  =SCANS / f"{scan_name}_seg.json"
tooth_seg_labels = load_segmentation(seg_path)

#### funzioni di pertbuazione

In [145]:
import random
import numpy as np

#Rimuove una percentuale di landmark
def apply_missing_landmark_perturbation(
    coords,
    classes,
    fraction,
    #specifica opzionalmente un sotto-insieme di classi da considerare. Es. Mesial
    target_landmark_classes=[],
    seed=42,
):
    rng = random.Random(seed)
    if len(coords) == 0:
        return coords.copy(), classes.copy()
    if len(target_landmark_classes)==0:
        candidate_idx = list(range(len(coords)))
    else:
        candidate_idx = [i for i, cls in enumerate(classes) if cls in target_landmark_classes]
    if len(candidate_idx) == 0:
        return ( coords.copy(),classes.copy() )
    n_remove = max( 1, int(len(candidate_idx) * fraction))

    remove_idx = set(rng.sample(  candidate_idx, min(    n_remove,len(candidate_idx))) )
    keep_idx = [i for i in range(len(coords)) if i not in remove_idx]
    coords_new = coords[keep_idx]
    classes_new = [classes[i] for i in keep_idx]
    return (
        coords_new,
        classes_new,
    )

In [133]:
coords_pred, classes_pred = load_pred_landmarks(PRED_CSV, scan_name)
coords_gt, classes_gt = load_gt_landmarks(gt_json)

noisy_coords, noisy_classes=apply_missing_landmark_perturbation(
    coords_pred,
    classes_pred,
    fraction=0.990,
    target_landmark_classes='Mesial',
    seed=42)
mesh = load_mesh(mesh_path)
mesh = color_mesh_by_groups(mesh, tooth_seg_labels, TOOTH_TO_GROUP, TOOTH_GROUP_PALETTE)
spheres_pred = create_landmark_spheres(coords_pred, classes_pred, LANDMARK_PALETTE)
spheres_gt = create_landmark_spheres(coords_gt, classes_gt, LANDMARK_PALETTE)

spheres_noisy = create_landmark_spheres(noisy_coords, noisy_classes, LANDMARK_PALETTE)

#visualize(mesh, spheres_pred, view="front")
#visualize(mesh, spheres_noisy, view="front")
scan_evaluation_=evaluate_single_scan_from_landmarks(scan_name=scan_name,
                                    coords_pred=coords_pred,
                                    classes_pred=classes_pred,
                                    coords_gt=coords_gt,
                                    classes_gt=classes_gt,categories=CATEGORIES)
scan_quality_profile=quality_profile_for_scan([scan_evaluation_],categories=CATEGORIES,scan_name=scan_name)


In [88]:
scan_evaluation_

{'scan': 'RZ69JRI9_lower',
 'ap_per_class': {'Mesial': 0.7710164835164836,
  'Distal': 0.8829204745871413,
  'Cusp': 0.672600358049556,
  'InnerPoint': 0.6462125220458553,
  'OuterPoint': 0.7740656565656566,
  'FacialPoint': 0.7302819865319867},
 'ar_per_class': {'Mesial': 0.5785714285714285,
  'Distal': 0.30833333333333346,
  'Cusp': 0.3454545454545451,
  'InnerPoint': 0.5249999999999999,
  'OuterPoint': 0.5833333333333337,
  'FacialPoint': 0.5000000000000002},
 'gt_count_per_class': {'Mesial': 14,
  'Distal': 12,
  'Cusp': 22,
  'InnerPoint': 12,
  'OuterPoint': 12,
  'FacialPoint': 12},
 'pred_count_per_class': {'Mesial': 14,
  'Distal': 12,
  'Cusp': 27,
  'InnerPoint': 10,
  'OuterPoint': 12,
  'FacialPoint': 11},
 'mAP': 0.7461829135494465,
 'mAR': 0.4734487734487735}

In [ ]:
 {'scan': 'RZ69JRI9_lower', 'ap_per_class': {'Mesial': 0.1333333333333333, 'Distal': 0.8829204745871413, 'Cusp': 0.672600358049556, 'InnerPoint': 0.6462125220458553, 'OuterPoint': 0.7740656565656566, 'FacialPoint': 0.7302819865319867}, 'ar_per_class': {'Mesial': 0.028571428571428595, 'Distal': 0.30833333333333346, 'Cusp': 0.3454545454545451, 'InnerPoint': 0.5249999999999999, 'OuterPoint': 0.5833333333333337, 'FacialPoint': 0.5000000000000002}, 'gt_count_per_class': {'Mesial': 14, 'Distal': 12, 'Cusp': 22, 'InnerPoint': 12, 'OuterPoint': 12, 'FacialPoint': 12}, 'pred_count_per_class': {'Mesial': 2, 'Distal': 12, 'Cusp': 27, 'InnerPoint': 10, 'OuterPoint': 12, 'FacialPoint': 11}, 'mAP': 0.6399023885189216, 'mAR': 0.38178210678210683}


In [ ]:
scan_evaluation_.keys()

dict_keys(['scan', 'ap_per_class', 'ar_per_class', 'gt_count_per_class', 'pred_count_per_class', 'mAP', 'mAR'])

In [146]:
def apply_perturbation(perturbation_type,coords_pred,classes_pred,perturbation_params):
    if perturbation_type == "missing_landmarks":
        return apply_missing_landmark_perturbation(
            coords_pred,
            classes_pred,
            fraction=perturbation_params["fraction"],
            target_landmark_classes=perturbation_params["target_landmark_classes"],
        )
    return coords_pred,classes_pred


In [ ]:
def build_synthetic_oracle_sample(
    scan_name,
    mesh_path,
    gt_path,
    seg_path,
    pred_landmark_path,
    screenshot_dir,
    perturbation_type,
    perturbation_params,
    prefix=''
):

    pert_scan_name = f"{scan_name}_{perturbation_type}_{build_perturbation_label(perturbation_params)}"
    tooth_seg_labels = load_segmentation(seg_path)
    mesh = load_mesh(mesh_path)
    mesh = color_mesh_by_groups(mesh, tooth_seg_labels, TOOTH_TO_GROUP, TOOTH_GROUP_PALETTE) 

    coords_pred, classes_pred = load_pred_landmarks(pred_landmark_path, scan_name)
    #print('## len class pred',len(classes_pred),'###shape coords pred==>',coords_pred.shape)
    #print('### pred landmark path ==>')
    coords_gt, classes_gt = load_gt_landmarks(gt_path)

    #applica pertubazione richista
    coords_after_pert,classes_after_pert=apply_perturbation(perturbation_type,coords_pred,classes_pred,perturbation_params)
    #print('pert scan name==>',pert_scan_name)
    #print("original:", len(classes_pred))
    #print("noisy:", len(classes_after_pert))
    #valuta scan e ottieni profilo qualità
    scan_evaluation_pred=evaluate_single_scan_from_landmarks(scan_name=scan_name,
                                coords_pred=coords_after_pert,
                                classes_pred=classes_after_pert,
                                coords_gt=coords_gt,
                                classes_gt=classes_gt,categories=CATEGORIES)
    #print('####### scan eval pred==>\n',scan_evaluation_pred)
    scan_quality_profile_pred=quality_profile_for_scan([scan_evaluation_pred],categories=CATEGORIES,scan_name=scan_name)
    #crea screenshot 
    spheres_pred_after_pert = create_landmark_spheres(coords_after_pert, classes_after_pert, LANDMARK_PALETTE)
    image_pred_after_pert_path = screenshot_dir / scan_name /f"{pert_scan_name}.png"
    save_screenshot(mesh, spheres_pred_after_pert, image_pred_after_pert_path, zoom=0.65)


    obs_features ={
                f'{prefix}_scan_conf_name': pert_scan_name,
                f'{prefix}_count_per_class':scan_evaluation_pred.get('pred_count_per_class'), 
                f'{prefix}_quality_score':scan_quality_profile_pred.get('quality_global'),
                f'{prefix}_gmap': scan_evaluation_pred.get('mAP'),
                f'{prefix}_quality_per_class': scan_quality_profile_pred.get('quality_per_class'), 
                f'{prefix}_metadata': perturbation_params.get('metadata',{}),
                f'{prefix}_image_path': image_pred_after_pert_path
            }
    return obs_features

In [169]:
def build_missing_landmark_configs():

    configs = []

    for frac, severity in [
        (0.05, "low"),
        (0.1, "medium-low"),
    ]:
        configs.append({
            "type": "missing_landmarks",
            "params": {
                "target_landmark_classes":[],
                "fraction": frac},
            "metadata": {
                "severity": severity,
                "description":
                    f"removed {int(frac*100)}% landmarks globally" }
        })

    #configurazioni rispetto ad una classe target
    for cls in [
        "Mesial",
        "Distal",
        "InnerPoint",
        "OuterPoint",
    ]:
        for frac, severity in [ (0.30, "medium"),(0.60, "high")]:
            configs.append({
                "type": "missing_landmarks",
                "params": {
                    "fraction": frac, 
                    "target_landmark_classes": [] if cls == 'any class' else  [cls] },
                "metadata": {
                    "severity": severity,
                    "description":
                        f"rimosso {int(frac*100)}% {cls}" }
            })
    return configs

In [170]:
PERTURBATIONS=[] 
PERTURBATIONS.extend(build_missing_landmark_configs())

In [171]:
len(PERTURBATIONS)

10

In [172]:
#qui potrei aggiungere anche un max di campioni che hanno specifica qualità (es. quality=3 o quality=4) prima di passare
#ad un'altra configurazione
def build_synthetic_oracle_dataset(scan_names, SCANS, GT_ROOT, PRED_CSV,SCREENSHOT_DIR,pertubation_configs):
    dataset = []
    for scan_name in scan_names: 
        mesh_path = SCANS / f"{scan_name}.obj"
        gt_path = GT_ROOT / f"{scan_name}__kpt.json"
        seg_path  =SCANS / f"{scan_name}_seg.json"

        #memorizziamo le informazioni basilari sullinput sul quale viene applicata la pertubazione.
        #in modo da fornire all'agente contesto maggiore sul dato originale e non solo sulla versione pertubata
        pred_features_before_pert=build_synthetic_oracle_sample(scan_name,
                                    mesh_path,gt_path,seg_path,
                                    PRED_CSV,SCREENSHOT_DIR,
                                    perturbation_type='',
                                    perturbation_params={},
                                    prefix='origin_pred')
        #print('##### PRED INFO BEFORE PERTUBATION==>\n',pred_features_before_pert)

        #applica tutte le possibili pertubazioni per generare k dati sintetici
        for pert_cfg in pertubation_configs: 
            pert_features=build_synthetic_oracle_sample(scan_name,
                                                mesh_path,gt_path,seg_path,
                                                PRED_CSV,SCREENSHOT_DIR,
                                                perturbation_type=pert_cfg.get('type'),
                                                perturbation_params=pert_cfg.get('params'),
                                                prefix='pred')
            dataset.append({
            "scan": scan_name,
            **pred_features_before_pert, 
            **pert_features,
            "mesh": mesh_path,
            "gt": gt_path,
            "seg": seg_path,
        })
        return dataset

In [173]:
scan_names=['F92OKIOI_upper', 'K4BAII5F_upper', 'RZ69JRI9_lower']
oracle_dataset=build_synthetic_oracle_dataset([scan_names[2]], SCANS, GT_ROOT, PRED_CSV,SCREENSHOT_DIR,pertubation_configs=PERTURBATIONS)

pert scan name==> RZ69JRI9_lower__
original: 86
noisy: 86
Screenshot salvato in: c:\Users\aless\Tesi_Msc\multi_agent_evaluation_on_intraoral_3D_scans\dataset\screenshot_scans\raw\RZ69JRI9_lower\RZ69JRI9_lower__.png
pert scan name==> RZ69JRI9_lower_missing_landmarks_target_landmark_classes___fraction_0.05
original: 86
noisy: 82
Screenshot salvato in: c:\Users\aless\Tesi_Msc\multi_agent_evaluation_on_intraoral_3D_scans\dataset\screenshot_scans\raw\RZ69JRI9_lower\RZ69JRI9_lower_missing_landmarks_target_landmark_classes___fraction_0.05.png
pert scan name==> RZ69JRI9_lower_missing_landmarks_target_landmark_classes___fraction_0.1
original: 86
noisy: 78
Screenshot salvato in: c:\Users\aless\Tesi_Msc\multi_agent_evaluation_on_intraoral_3D_scans\dataset\screenshot_scans\raw\RZ69JRI9_lower\RZ69JRI9_lower_missing_landmarks_target_landmark_classes___fraction_0.1.png
pert scan name==> RZ69JRI9_lower_missing_landmarks_fraction_0.3__target_landmark_classes_Mesial
original: 86
noisy: 82
Screenshot sal

In [175]:
oracle_dataset[0].keys()

dict_keys(['scan', 'origin_pred_scan_conf_name', 'origin_pred_count_per_class', 'origin_pred_quality_score', 'origin_pred_gmap', 'origin_pred_quality_per_class', 'origin_pred_metadata', 'origin_pred_image_path', 'pred_scan_conf_name', 'pred_count_per_class', 'pred_quality_score', 'pred_gmap', 'pred_quality_per_class', 'pred_metadata', 'pred_image_path', 'mesh', 'gt', 'seg'])

In [174]:
for scan_ in oracle_dataset:
    print('#######')
    print('scan==>',scan_['scan'])

    print('pertubazione name=>',scan_['pred_scan_conf_name'])
    print('quality origin==>',scan_['origin_pred_quality_score'])
    print('quality pred==>',scan_['pred_quality_score'])
    print('maAP origin==>',scan_['origin_pred_gmap'])
    print('maAP pred==>',scan_['pred_gmap'])


#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landmarks_target_landmark_classes___fraction_0.05
quality origin==> 5
quality pred==> 4
maAP origin==> 0.7461829135494465
maAP pred==> 0.7051476661198882
#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landmarks_target_landmark_classes___fraction_0.1
quality origin==> 5
quality pred==> 3
maAP origin==> 0.7461829135494465
maAP pred==> 0.682666884919676
#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landmarks_fraction_0.3__target_landmark_classes_Mesial
quality origin==> 5
quality pred==> 4
maAP origin==> 0.7461829135494465
maAP pred==> 0.7083768153266816
#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landmarks_fraction_0.6__target_landmark_classes_Mesial
quality origin==> 5
quality pred==> 3
maAP origin==> 0.7461829135494465
maAP pred==> 0.6713309599474929
#######
scan==> RZ69JRI9_lower
pertubazione name=> RZ69JRI9_lower_missing_landma